In [ ]:
# 🔹 1. SETUP
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

# 🔹 2. CONFIG
TRAIN_DIR = "/kaggle/input/competitions/sign-language-contest/train"
TEST_DIR  = "/kaggle/input/competitions/sign-language-contest/test"
SAMPLE_SUB = "/kaggle/input/competitions/sign-language-contest/sample_submission.csv"

IMG_SIZE = 224
BATCH = 32  # If you face a GPU OOM later, reduce this to 16
EPOCHS = 25

# 🔹 3. GPU STRATEGY
strategy = tf.distribute.MirroredStrategy()
print("GPUs:", strategy.num_replicas_in_sync)

# 🔹 4. DATA LOADING (Kept raw 0-255 values here)
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.15,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.15,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    label_mode='categorical'
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

# ❌ REMOVED .cache() to prevent the System RAM crash you experienced
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

# 🔹 5. SAFE AUGMENTATION (BEST FOR ASL)
aug = models.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomTranslation(0.10, 0.10),
    layers.RandomZoom(0.12),
    layers.RandomContrast(0.20),
    layers.RandomBrightness(0.12),
])

# 🔹 6. CALLBACKS
def get_callbacks(name):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            f"best_{name}.keras",
            monitor="val_accuracy",
            save_best_only=True,
            mode="max",
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            verbose=1
        )
    ]

# 🔹 7. MODEL BUILDER
def build_model(base_fn, include_rescaling=False, freeze=50):
    base = base_fn(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    base.trainable = True
    for layer in base.layers[:-freeze]:
        layer.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = aug(inputs)

    # Explicit rescaling wrapper for models that need it (like MobileNetV2)
    if include_rescaling:
        x = layers.Rescaling(1./127.5, offset=-1.0)(x)

    x = base(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        256,
        activation='relu',
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)

    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation='softmax'
    )(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )

    return model

# 🔹 8. MODEL 1 (MobileNetV2 - Requires rescaling)
with strategy.scope():
    model1 = build_model(
        tf.keras.applications.MobileNetV2,
        include_rescaling=True,
        freeze=40
    )

history1 = model1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=get_callbacks("mobilenet")
)

# 🔹 9. MODEL 2 (EfficientNetB0 - Built-in rescaling)
with strategy.scope():
    model2 = build_model(
        tf.keras.applications.EfficientNetB0,
        include_rescaling=False,
        freeze=60
    )

history2 = model2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=get_callbacks("efficientnet")
)

# 🔹 10. TEST PIPELINE
sample_df = pd.read_csv(SAMPLE_SUB)
test_files = sample_df['image_id'].tolist()

test_paths = [os.path.join(TEST_DIR, f) for f in test_files]

def load_img(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img

test_ds = tf.data.Dataset.from_tensor_slices(test_paths)
test_ds = test_ds.map(load_img).batch(BATCH).prefetch(tf.data.AUTOTUNE)

# 🔹 11. PREDICTIONS
p1 = model1.predict(test_ds)
p2 = model2.predict(test_ds)

# 🔹 VECTORIZED TTA FUNCTIONS FOR BATCHES
def tta_contrast(batch):
    return tf.image.adjust_contrast(batch, 1.1)

def tta_brightness(batch):
    return tf.image.adjust_brightness(batch, 0.08)

tta_c = test_ds.map(tta_contrast, num_parallel_calls=tf.data.AUTOTUNE)
tta_b = test_ds.map(tta_brightness, num_parallel_calls=tf.data.AUTOTUNE)

p1_tta = model1.predict(tta_c)
p2_tta = model2.predict(tta_b)

# 🔹 ENSEMBLE
final_pred = (
    0.40 * p1 +
    0.40 * p2 +
    0.10 * p1_tta +
    0.10 * p2_tta
)

# 🔹 12. SUBMISSION
pred_idx = np.argmax(final_pred, axis=1)
pred_labels = [class_names[i] for i in pred_idx]

submission = pd.DataFrame({
    "image_id": test_files,
    "label": pred_labels
})

submission = sample_df[['image_id']].merge(submission, on='image_id', how='left')
submission['label'] = submission['label'].fillna(class_names[0])

submission.to_csv("submission.csv", index=False)

print("DONE ✔ Final submission ready")
print(submission.head())

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

print("⚡ Running Final Leaderboard Pipeline...")

# Hardcoded directly to the folder visible on your screen
TEST_DIR = "/kaggle/input/competitions/sign-language-contest/test"
SAMPLE_SUB = "/kaggle/input/competitions/sign-language-contest/sample_submission.csv"

print(f"📁 Target Test Directory: {TEST_DIR}")
print(f"📄 Target Sample Submission: {SAMPLE_SUB}")

IMG_SIZE = 224
BATCH = 32

# Load sample submission
sample_df = pd.read_csv(SAMPLE_SUB)
test_files = sample_df['image_id'].tolist()
test_paths = [os.path.join(TEST_DIR, f) for f in test_files]

def load_img(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) # Crucial float conversion for TTA math
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img

# Set up optimized test data pipelines
test_ds = tf.data.Dataset.from_tensor_slices(test_paths)
test_ds = test_ds.map(load_img).batch(BATCH).prefetch(tf.data.AUTOTUNE)

def tta_contrast(batch):
    return tf.image.adjust_contrast(batch, 1.1)

def tta_brightness(batch):
    return tf.image.adjust_brightness(batch, 0.08)

tta_c = test_ds.map(tta_contrast, num_parallel_calls=tf.data.AUTOTUNE)
tta_b = test_ds.map(tta_brightness, num_parallel_calls=tf.data.AUTOTUNE)

# 1. Load the pre-saved 99.9% MobileNetV2 Weights
with strategy.scope():
    fast_model1 = build_model(tf.keras.applications.MobileNetV2, include_rescaling=True, freeze=40)
    
    if os.path.exists("best_mobilenet.keras"):
        fast_model1.load_weights("best_mobilenet.keras")
        print("✔ Loaded best MobileNetV2 weights successfully!")
    else:
        print("⚠️ Warning: best_mobilenet.keras not found, using current weights.")

# 2. Predict using TTA Ensemble blends to smash background variations
print("🏃 Running predictions on test dataset...")
p1 = fast_model1.predict(test_ds)

print("✨ Running background-robust TTA adjustments...")
p1_tta_c = fast_model1.predict(tta_c)
p1_tta_b = fast_model1.predict(tta_b)

# Final robust blend
final_pred = (0.70 * p1) + (0.15 * p1_tta_c) + (0.15 * p1_tta_b)

# 3. Compile Final Submission Dataframe
pred_idx = np.argmax(final_pred, axis=1)
pred_labels = [class_names[i] for i in pred_idx]

submission = pd.DataFrame({
    "image_id": test_files,
    "label": pred_labels
})

submission = sample_df[['image_id']].merge(submission, on='image_id', how='left')
submission['label'] = submission['label'].fillna(class_names[0])

# Export directly to working directory
submission.to_csv("submission.csv", index=False)

print("\n🎉 SUCCESS! submission.csv has been completely generated!")
print(submission.head())